教学演示 思维链

核心学习目标

掌握任务语义解析、事实状态推理、信息完备性分析，理解复杂任务中的显式推理链构建机制

设计

输入：用户请求
展示使用思维链推理出 已知、查询可知、需向用户澄清事实等

**本课两个重点**：
1. 用 **LangChain `PromptTemplate`** 生成与 `str.format(task=…)` 相同的完整提示词，并**展示全文**（教学演示「思维链提示词长什么样」）。
2. （可选）用 **`ChatOpenAI`** 发一条请求，看模型按四段标题（已给出或已验证的事实 / 需要查阅的事实 / 需要推导的事实 / 有根据的猜测）回复。


## 0. 环境

```bash
pip install langchain-core langchain-openai httpx
```

若最后一格要真调模型，请设置：`OPENAI_API_KEY`、`OPENAI_BASE_URL`、`OPENAI_MODEL`（或在代码里临时写死）。

In [1]:
# Windows 下避免 print 长中文/特殊符号时报编码错（可选）
import sys

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

## 1. 思维链提示词模板

占位符只有一个：`{task}` → 替换为用户的自然语言任务。

In [3]:
FACTS_PROMPT = """下面我将向您提出一个请求。在开始处理该请求之前，请先回答以下预调查问题，尽您所能作答。请记住，您在 trivia（常识问答）方面具有 Ken Jennings 级别的水平，在谜题方面具有 Mensa（门萨俱乐部）级别的水平，因此应该有丰富的知识储备可供挖掘。

以下是该请求：

{task}

以下是预调查问题：

    1. 请列出请求中明确给出的任何具体事实或数据。可能没有任何此类信息。
    2. 请列出可能需要查阅的任何事实，以及具体可以在哪里找到这些信息。在某些情况下，请求本身会提及权威来源。
    3. 请列出可能需要推导的任何事实（例如，通过逻辑演绎、模拟或计算得出）。
    4. 请列出从记忆中回忆出的任何事实、直觉、经过充分推理的猜测等。

在回答此调查时，请记住，"事实"通常是具体的名称、日期、统计数据等。您的回答应使用以下标题：

    1. 已给出或已验证的事实
    2. 需要查阅的事实
    3. 需要推导的事实
    4. 有根据的猜测

不要在您的回复中包含其他任何标题或部分。在被要求之前，不要列出下一步行动或计划。
"""

# 课堂示例任务
task = """我要去北京玩三天"""

## 2. 用 LangChain 生成「完整提示词」

`PromptTemplate.from_template` 与 Python 的 `FACTS_PROMPT.format(task=…)` 在这里**等价**，便于后面接 LCEL、链式调试等。

In [4]:
from langchain_core.prompts import PromptTemplate

facts_template = PromptTemplate.from_template(FACTS_PROMPT)
full_prompt = facts_template.format(task=task.strip())

# 与纯 format 一致（可自行对照）
assert full_prompt == FACTS_PROMPT.format(task=task.strip())

C:\Users\zhanghong26\.conda\envs\enterprise_bench_agents_test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. 展示：拼好后的整条「思维链」提示词

下面就是编排器第一步要发给模型的**完整用户消息**（节选可在课堂改成 `[:2000]` 等）。

In [5]:
from IPython.display import display, Markdown

display(Markdown(f"**字符数：** {len(full_prompt)}"))
display(Markdown("```text\n" + full_prompt + "\n```"))

**字符数：** 492

```text
下面我将向您提出一个请求。在开始处理该请求之前，请先回答以下预调查问题，尽您所能作答。请记住，您在 trivia（常识问答）方面具有 Ken Jennings 级别的水平，在谜题方面具有 Mensa（门萨俱乐部）级别的水平，因此应该有丰富的知识储备可供挖掘。

以下是该请求：

我要去北京玩三天

以下是预调查问题：

    1. 请列出请求中明确给出的任何具体事实或数据。可能没有任何此类信息。
    2. 请列出可能需要查阅的任何事实，以及具体可以在哪里找到这些信息。在某些情况下，请求本身会提及权威来源。
    3. 请列出可能需要推导的任何事实（例如，通过逻辑演绎、模拟或计算得出）。
    4. 请列出从记忆中回忆出的任何事实、直觉、经过充分推理的猜测等。

在回答此调查时，请记住，"事实"通常是具体的名称、日期、统计数据等。您的回答应使用以下标题：

    1. 已给出或已验证的事实
    2. 需要查阅的事实
    3. 需要推导的事实
    4. 有根据的猜测

不要在您的回复中包含其他任何标题或部分。在被要求之前，不要列出下一步行动或计划。

```

## 4.（可选）LangChain `ChatOpenAI` 调用一次

把上格生成的 `full_prompt` 作为**一条用户消息**发给 OpenAI 兼容网关即可。下面用环境变量；课堂演示也可改成与你 `ChatOpenAI(model=..., base_url=..., api_key=...)` 一样的字面量。

In [6]:
import os

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

kw = dict(
        model="qwen",
        temperature=0,
        base_url="model base_url",
        api_key="your api key ",
)
import httpx
kw["http_client"] = httpx.Client(verify=False)
llm = ChatOpenAI(**kw)
resp = llm.invoke([HumanMessage(content=full_prompt)])
display(Markdown("### 模型输出"))
display(Markdown(resp.content if isinstance(resp.content, str) else str(resp.content)))

### 模型输出

**1. 已给出或已验证的事实**  
- 您计划在北京停留三天。  

**2. 需要查阅的事实**  
- 北京主要旅游景点的开放时间、门票价格及是否需要提前预约（如故宫、颐和园、天坛等）。  
- 北京公共交通（地铁、公交）的线路图、运营时间及票价，尤其是从机场/火车站到市中心的交通方式。  
- 当地天气预报（2026 年 5 月中下旬）以便安排室内外活动。  
- 北京的热门美食街或餐厅的营业时间、推荐菜品及是否需要排队或预约。  
- 近期北京是否有大型活动、展览或节庆（如演唱会、体育赛事）可能影响人流或交通。  
- 近期北京的疫情防控政策（如核酸检测、健康码要求）以及入境/出境的相关规定（如适用）。  

**3. 需要推导的事实**  
- 根据三天的时间，合理的每日行程安排（每日至少覆盖多少主要景点才能兼顾深度与休息）。  
- 从主要景点之间的距离与地铁/公交换乘时间，推算出最省时的路线顺序。  
- 估算每日的交通费用、门票总支出以及餐饮预算，以制定大致的旅行费用预算。  
- 根据天气预报的温度与降雨概率，推断需要携带的衣物和雨具。  

**4. 有根据的猜测**  
- 北京的春季（5 月）天气通常温暖舒适，日均气温在 20‑26°C 左右，降雨概率适中，适合步行游览。  
- 三天的行程若以经典景点为主，可能会包括：天安门广场、故宫、景山公园、北海公园、颐和园、圆明园、天坛、后海/什刹海地区以及王府井或南锣鼓巷的美食街。  
- 若使用地铁作为主要交通方式，预计每日交通费用在 30‑50 元人民币之间（含机场快线或城际铁路的往返费用）。  
- 餐饮方面，普通餐厅人均约 50‑100 元，高档餐厅或特色餐饮人均可达 200 元以上。  
- 若提前在官方渠道（如故宫官网）预约门票，可避免现场排队，提升游览效率。

## 5. 小结

- **思维链提示词**在这里 = 固定英文说明 + 嵌入的 `{task}` + 要求模型按四段标题作答；**全部放在一条 user 内容里**。
- **LangChain**：`PromptTemplate` 负责字符串拼装；`ChatOpenAI` 负责调用；